In [86]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from termcolor import cprint
import results_analysis_utils as rutils
from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
pd.set_option('display.max_columns', 100)


%load_ext autoreload
%autoreload 2

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['figure.dpi'] = 100 
plt.rcParams['savefig.dpi'] = 100

# ===================================================================
# 1. GROUND TRUTH DICTIONARIES
# ===================================================================

single_fish_gt = {
    "2024_11_12": 31.5, "2024_11_28": 28.9, 
    "2025_05_08": 33.5, "2025_08_21": 29.1
    
}

fish_measures = {
    "Fish_0": 31.5, "Fish_1": 28.9, "Fish_2": 21.6,
    "Fish_3": 33.5, "Fish_5": 30.5, "Fish_4": 33.5, 
    "Fish_5": 30.5, "not_a_fish": -100, "None": -100,
    "Red_tag":29.1, "Black_tag":26.6, "unnmarked":32.3
}

reverse_fish_measures = {v: k for k, v in fish_measures.items()}

global_single_dfs = []
global_multi_dfs = []


multiple_fish_gt = {
        
    "2024_11_28": {
        '13-14-43_0': {1: "Fish_1",2: "not_a_fish", 4: "not_a_fish", 8: "Fish_2", 11: "Fish_2", 21: "Fish_2",23:"Fish_1", 24: "Fish_1",
                       29: "Fish_2", 30:"not_a_fish",31: "None", 
                       33:"not_a_fish",34: "None", 37: "Fish_1",38:"not_a_fish",40: "None", 45: "Fish_1",46: "None", 51: "None", 
                       58: "Fish_2",59: "None",
                       61: "Fish_2", 62: "Fish_2", 65: "None", 66: "None", 75: "Fish_2", 77: "Fish_2", 81: "Fish_2", 82: "Fish_2",
                       2: "not_a_fish", 4: "not_a_fish", 8: "None", 11: "Fish_2", 31: "not_a_fish", 34: "not_a_fish", 40: "Fish_2", 46: "Fish_2", 51: "not_a_fish", 59: "Fish_2", 65: "not_a_fish", 66: "not_a_fish"},
        '13-16-54_0': {1: "Fish_1", 5: "Fish_2", 7:"not_a_fish", 12: "Fish_2", 13: "Fish_2",17: "Fish_2",19: "Fish_2", 29: "Fish_2", 41: "Fish_1", 49: "Fish_1", 53: "Fish_1", 55: "Fish_2",59: "Fish_2", 67: "Fish_2", 68: "Fish_2", 70: "Fish_1", 71: "Fish_2",4: "Fish_2", 9: "Fish_2", 15: "Fish_2", 39: "Fish_2", 40: "Fish_2", 48: "Fish_2", 74: "Fish_1"},
        '13-27-52_0': {16: "Fish_2", 36: "Fish_2", 40: "not_a_fish", 1: "Fish_1", 2: "Fish_2", 3: "Fish_2",4:"not_a_fish", 5: "Fish_2", 6: "Fish_2", 8: "Fish_2", 11: "Fish_1", 12: "Fish_2", 13: "Fish_2",30:"not_a_fish", 33: "Fish_1", 35: "not_a_fish", 37: "Fish_1"},
        '13-32-16_0': {3: "not_a_fish", 18: "Fish_2", 19: "Fish_2", 24: "Fish_2", 26: "not_a_fish", 32: "Fish_1", 38: "Fish_1", 39: "Fish_1", 40: "Fish_1",1: "Fish_2", 4: "Fish_1", 6: "Fish_1", 17: "Fish_2", 25: "Fish_1", 28: "Fish_1", 30: "Fish_1",35:"Fish_1", 44: "Fish_1", 52: "Fish_1", 53: "Fish_1", 67: "Fish_1"},
        '13-34-27_0': {14: "Fish_2", 17: "Fish_2", 24: "not_a_fish",1: "Fish_1", 5: "Fish_1", 8: "Fish_1", 16: "Fish_1", 21: "Fish_1", 23: "Fish_2", 28: "Fish_1", 30: "not_a_fish", 31: "Fish_1"},
        '13-36-49_0': {28: "not_a_fish", 36: "not_a_fish",1: "Fish_2", 2: "Fish_1", 3: "Fish_2", 6: "Fish_2",7:"Fish_2", 10: "Fish_2", 11: "Fish_2", 19: "Fish_2", 20: "Fish_1", 24: "Fish_2", 30: "Fish_1", 33: "Fish_2", 40: "not_a_fish", 41: "not_a_fish", 42: "not_a_fish"},
        '13-39-00_0': {16: "not_a_fish", 17: "not_a_fish", 18: "not_a_fish", 20: "not_a_fish", 26: "Fish_2",1: "not_a_fish", 2: "Fish_1", 8: "Fish_1", 11: "Fish_1", 12: "Fish_1",13:"not_a_fish", 23: "not_a_fish"},
        '13-41-12_0': {1: "not_a_fish", 7: "Fish_1", 8: "not_a_fish", 19: "Fish_2",2: "Fish_2", 3: "Fish_1", 11: "Fish_1",14:"Fish_2"},
        '13-42-45_0': {12: "not_a_fish", 13: "Fish_1", 15: "not_a_fish", 21: "not_a_fish", 25: "not_a_fish", 28: "not_a_fish",1: "Fish_1", 2: "Fish_2", 5: "Fish_1",8:"not_a_fish", 17: "Fish_1", 26: "Fish_1"},
        '13-48-24_0': {2: "Fish_2", 12: "not_a_fish", 28: "Fish_1", 41: "not_a_fish", 43: "Fish_1", 44: "Fish_2",1: "Fish_1", 4: "Fish_2", 5: "Fish_2", 10: "Fish_2"},
        '13-49-20_0': {1: "Fish_1", 2: "Fish_2", 9: "Fish_2", 13: "Fish_1", 16: "Fish_2", 25: "Fish_1", 29: "Fish_2", 32: "Fish_2", 34: "Fish_2", 36: "Fish_2", 41: "Fish_2", 44: "fish_1", 56: "fish_1", 58: "Fish_2", 60: "fish_1", 61: "Fish_2", 64: "Fish_2", 69: "fish_1", 72: "fish_1", 73: "Fish_1", 75: "fish_1", 76: "fish_1", 77: "fish_1"},
        "13-19-06_0": {13: "Fish_2", 18: "Fish_2", 34: "not_a_fish", 53: "not_a_fish",1: "Fish_1", 5: "Fish_2", 7: "Fish_1", 12: "Fish_1", 15: "Fish_2", 19: "Fish_2", 22: "Fish_1", 25: "Fish_2", 27: "Fish_2", 33: "Fish_2", 40: "Fish_2"},
        "13-21-17_0": {3: "not_a_fish", 7: "not_a_fish", 13: "not_a_fish", 14: "not_a_fish", 36: "Fish_1", 
                       38: "Fish_1", 42: "not_a_fish", 67: "Fish_1", 71: "not_a_fish", 73: "Fish_2", 75: "not_a_fish", 79: "not_a_fish",1: "Fish_2", 2: "Fish_1", 4: "Fish_2", 15: "not_a_fish", 21: "Fish_1", 24: "Fish_2", 25: "not_a_fish", 30: "not_a_fish", 32: "Fish_1", 33: "Fish_1", 37: "Fish_2", 39: "Fish_2", 60: "Fish_2", 62: "Fish_1", 65: "Fish_1", 68: "Fish_1", 69: "None", 70: "not_a_fish", 72: "Fish_2", 78: "Fish_2"},
        "13-23-29_0": {2: "not_a_fish", 3: "not_a_fish", 7: "not_a_fish", 
                       8: "not_a_fish", 10: "not_a_fish", 17: "not_a_fish", 29: "not_a_fish", 33: "Fish_2", 57: "Fish_1", 59: "not_a_fish",1: "Fish_2", 4: "Fish_2", 12: "Fish_2", 15: "Fish_2", 16: "Fish_1",19: "Fish_1", 21: "Fish_2", 25: "Fish_2", 27: "Fish_1", 32: "Fish_2",35: "Fish_2", 36: "Fish_1", 41: "Fish_2", 42: "not_a_fish", 44: "Fish_1", 48: "Fish_1", 50: "Fish_2", 54: "Fish_1",55: "Fish_1", 58: "Fish_1", 60: "Fish_2", 61: "Fish_2", 62: "Fish_1"}, 
        "13-25-41_0": {13: "not_a_fish", 14: "not_a_fish", 15: "Fish_2", 25: "Fish_1", 26: "not_a_fish", 
                       34: "Fish_2", 39: "Fish_1", 48: "not_a_fish", 54: "not_a_fish", 56: "not_a_fish", 58: "Fish_2", 
                       72: "Fish_2", 74: "Fish_1", 78: "Fish_1", 81: "None", 83: "not_a_fish",1: "Fish_1", 2: "Fish_2", 5: "Fish_2", 11: "Fish_1", 12: "Fish_1", 17: "Fish_2", 18: "Fish_2", 22: "Fish_1", 24: "Fish_1", 33: "Fish_1", 35: "Fish_2", 36: "Fish_1", 38: "Fish_2", 43: "Fish_1",44: "Fish_1", 46: "Fish_2", 51: "Fish_2", 52: "not_a_fish", 53: "Fish_2", 63: "Fish_1", 67: "Fish_2", 68: "Fish_1", 70: "Fish_2", 76: "Fish_1"},
        "13-47-08_0": {6: "not_a_fish", 10: "Fish_2", 11: "Fish_2", 13: "Fish_2", 14: "Fish_2", 15: "Fish_2", 21: "Fish_2",1: "Fish_1", 2: "Fish_2", 8: "Fish_2", 17: "Fish_1", 25: "Fish_2"},
    
    },
    
    
    "2025_05_08": {
        '11-28-42_0': {1: "Fish_5", 2: "Fish_3", 5: "Fish_5"},
        '11-29-16_1': {3: "not_a_fish", 9: "not_a_fish",1: "Fish_3", 2: "Fish_5", 6: "Fish_5", 8: "Fish_5"},
        '11-29-49_2': {8: "not_a_fish", 9: "not_a_fish",1: "Fish_5", 2: "Fish_3", 6: "Fish_5", 7: "Fish_5",12: "not_a_fish",15:"not_a_fish",19:"not_a_fish"},
        '11-30-34_0': {9: "not_a_fish",1: "Fish_5", 6: "Fish_3"},
        '11-31-08_1': {1: "Fish_3", 2: "Fish_5", 3: "Fish_5", 8: "Fish_5"},
        '11-33-23_0': {5: "not_a_fish", 7: "Fish_5", 12: "not_a_fish", 13: "not_a_fish",1: "Fish_3", 2: "Fish_5",4: "not_a_fish", 10: "Fish_5", 18: "Fish_3", 19: "Fish_5", 20: "Fish_5"},
        '11-33-52_1': {1: "Fish_3", 2: "Fish_5", 4: "Fish_3"},
        '11-36-36_0': {9: "not_a_fish", 20: "Fish_5", 21: "Fish_5", 25: "not_a_fish", 29: "not_a_fish", 31: "Fish_5",1: "Fish_3", 2: "Fish_5", 8: "Fish_3", 13: "Fish_5", 23: "Fish_5", 27: "Fish_5", 30: "Fish_3"}, 
        '11-37-20_1': {9: "not_a_fish", 10: "not_a_fish", 11: "not_a_fish", 12: "Fish_5", 19: "not_a_fish", 21: "not_a_fish", 30: "not_a_fish",1: "not_a_fish", 2: "Fish_3", 8: "Fish_5", 16: "Fish_3", 18: "Fish_5", 25: "Fish_5", 33: "Fish_5", 34: "Fish_5"},
        '11-41-45_0': {3: "not_a_fish", 4: "not_a_fish", 10: "not_a_fish",1: "Fish_3", 8: "Fish_3"},
        '11-42-15_1': {1: "Fish_3"},
        '11-44-48_0': {1: "Fish_3", 3: "Fish_5", 6: "Fish_5", 7: "Fish_5", 10: "not_a_fish"},
        '11-48-15_0': {1: "Fish_3", 2: "Fish_5", 4: "Fish_4", 6: "Fish_4", 13: "Fish_4", 14: "Fish_3", 16: "Fish_5", 18: "Fish_4", 19: "Fish_4", 22: "Fish_4"},
        '11-49-52_0': {2: "not_a_fish",1: "Fish_3", 3: "Fish_5", 4: "Fish_4", 7: "Fish_4", 10: "Fish_3", 13: "Fish_5", 14: "Fish_4"},
        '11-57-17_0': {3: "not_a_fish", 19: "not_a_fish", 20: "not_a_fish",1: "Fish_3", 2: "Fish_4", 4: "Fish_5"},
        '11-57-59_0': {7: "Fish_5", 21: "not_a_fish", 27: "not_a_fish",1: "Fish_5", 2: "Fish_5", 4: "Fish_4", 9: "Fish_3", 10: "Fish_3", 16: "Fish_5", 20: "Fish_4", 24: "Fish_3"},
        '11-59-57_0': {10: "not_a_fish", 16: "Fish_5", 22: "Fish_3", 29: "not_a_fish", 34: "not_a_fish",1: "Fish_5", 2: "Fish_4", 8: "Fish_3", 14: "Fish_3", 20: "Fish_5", 24: "Fish_3"},
        '12-00-39_0': {20: "not_a_fish", 21: "not_a_fish", 23: "not_a_fish", 35: "not_a_fish", 44: "not_a_fish", 47: "not_a_fish", 48: "not_a_fish",1: "Fish_4", 2: "Fish_3", 3: "Fish_5", 14: "Fish_4", 17: "Fish_5", 29: "Fish_5", 34: "Fish_5", 36: "Fish_5", 39: "Fish_5"},
        '12-01-20_0': {21: "not_a_fish", 24: "not_a_fish", 30: "not_a_fish",1: "Fish_5", 2: "Fish_4", 3: "Fish_3", 4: "Fish_5", 5: "Fish_3", 9: "Fish_5", 13: "Fish_4", 26: "Fish_3", 29: "Fish_4", 33: "Fish_4"},
        '12-02-02_0': {4: "not_a_fish", 10: "not_a_fish",1: "Fish_4", 2: "Fish_5", 3: "Fish_3", 9: "Fish_5", 11: "Fish_4", 13: "Fish_4", 16: "Fish_5"},
        '12-02-44_0': {7: "Fish_5", 10: "Fish_4",1: "Fish_3", 2: "Fish_5", 3: "Fish_4", 11: "Fish_4", 13: "Fish_5"}
        
    
    },
    "2025_08_21": {
        "10-01-53_0_compressed-r201-end": {-1: "None", 1: "Red_tag", 3: "unnmarked", 4: "unnmarked", 10: "Red_tag", 
                                           11: "Red_tag", 13: "unnmarked", 18: "Red_tag", 19: "unnmarked", 20: "None", 
                                           31: "unnmarked", 32: "Red_tag"},
        
        "10-07-59_0-r50-end": {-1: "None", 1: "unnmarked", 2: "Red_tag", 3: "unnmarked", 
                               5: "unnmarked", 6: "Red_tag", 9: "unnmarked", 12: "unnmarked", 
                               13: "None", 14: "None", 15: "Red_tag", 16: "Red_tag", 
                               18: "Red_tag", 19: "Red_tag", 20: "Red_tag", 21: "unnmarked", 
                               22: "None", 24: "Red_tag", 27: "unnmarked", 28: "Red_tag"},
                
        "10-09-31_0-r0-344": {1: "unnmarked", 4: "Red_tag", 5: "unnmarked", 7: "unnmarked", 8: "None", 
                              9: "Red_tag", 10: "unnmarked", 12: "unnmarked", 16: "Red_tag", 17: "unnmarked"},

        "10-09-31_0-r398-end": {-1: "None", 1: "Red_tag", 2: "unnmarked", 4: "unnmarked", 
                                5: "unnmarked", 7: "Red_tag", 8: "unnmarked", 9: "unnmarked", 
                                12: "Red_tag", 13: "Red_tag", 14: "Red_tag"},
    
        "10-11-02_0-r0-244": {-1: "None", 1: "unnmarked", 2: "Red_tag", 
                              3: "Red_tag", 7: "unnmarked", 9: "None", 14: "Red_tag"},
    
        "10-11-02_0-r295-end": {1: "Red_tag", 2: "unnmarked", 4: "Red_tag", 
                                8: "None", 10: "Red_tag", 14: "unnmarked"},

        "10-12-34_0-r215-end": {1: "Red_tag", 2: "Red_tag", 4: "unnmarked", 
                                6: "Red_tag", 7: "None", 10: "Red_tag", 
                                12: "Red_tag", 14: "None", 15: "unnmarked"},
        "10-14-05_0-r0-528": {-1: "None", 1: "Red_tag", 2: "unnmarked", 6: "Red_tag", 
                              9: "unnmarked", 10: "Red_tag", 12: "None", 13: "unnmarked", 
                              15: "None", 16: "None", 17: "unnmarked", 18: "Red_tag"},
        
        "10-15-37_0-r496-end": {1: "unnmarked", 2: "Red_tag", 5: "Red_tag", 6: "unnmarked"},
        "10-18-24_0_compressed-r47-end": {-1: "None", 1: "Red_tag", 5: "unnmarked", 
                                          7: "unnmarked", 10: "Red_tag", 11: "None", 
                                          12: "Red_tag", 17: "Red_tag", 18: "Red_tag"},
        "10-19-44_0_compressed-r383-534": {1: "unnmarked", 2: "Red_tag"},
        "10-19-44_0_compressed-r588-end": {1: "Red_tag", 2: "unnmarked"},
        "10-20-29_1_compressed-r0-78": {-1: "None", 1: "Red_tag", 
                                        2: "unnmarked", 3: "unnmarked", 4: "Red_tag"},
        "10-22-00_1_compressed": {-1: "None", 1: "Red_tag", 2: "unnmarked",
                                  3: "Red_tag", 5: "None"},
        "10-37-15_1-r": {-1: "None", 1: "unnmarked", 2: "Red_tag", 
                         3: "Red_tag", 6: "Red_tag", 8: "unnmarked", 9: "not_a_fish", 10: "not_a_fish"},
        "13-05-55_0-r": {1: "Red_tag", 2: "unnmarked", 3: "Black_tag", 4: "Black_tag", 5: "unnmarked", 
                         8: "Black_tag", 10: "Red_tag", 11: "None", 15: "Black_tag", 20: "Red_tag", 
                         22: "None", 25: "None", 26: "unnmarked", 27: "None", 28: "unnmarked", 
                         30: "unnmarked", 34: "unnmarked", 38: "Red_tag", 39: "None", 42: "Red_tag", 
                         47: "unnmarked", 49: "Black_tag"},
        "13-22-22_0_compressed-r60-297": {1: "unnmarked", 2: "Black_tag", 3: "Red_tag", 6: "Red_tag", 
                                          7: "None", 9: "Red_tag", 11: "None", 13: "Black_tag"},
        "13-22-22_0_compressed-r378-463": {1: "Black_tag", 2: "unnmarked", 3: "Red_tag", 4: "not_a_fish"},
        "13-23-54_0_compressed-r0-304": {-1: "None", 2: "Red_tag", 3: "Black_tag", 5: "unnmarked"},
        "13-23-54_0_compressed-r450-524": {1: "unnmarked", 2: "Black_tag", 3: "Red_tag", 5: "None"},
        "13-23-54_0_compressed-r602-end": {1: "Red_tag", 2: "Black_tag", 3: "unnmarked"}
    }
     
}



ALL_DAYS = list(set(list(single_fish_gt.keys()) + list(multiple_fish_gt.keys())))
ASPECT_RATIO_THR = 3
ANGLE_THR = 30.0
csvs_not_generated = False
# ===================================================================
# 2. MASTER LOOP: ITERATE OVER BOTH DATASETS
# ===================================================================
if csvs_not_generated:
    for dataset_id in ["DATASET_1", "DATASET_2"]:
        cprint(f"\n\n{'='*70}", "white", "on_green", attrs=["bold"])
        cprint(f"🚀 STARTING AGGREGATION FOR: {dataset_id}", "white", "on_green", attrs=["bold"])
        cprint(f"{'='*70}", "white", "on_green", attrs=["bold"])
        
        BASE_DIR = Path(f"/home/slimbook/fish_sizing/ARTICLE/{dataset_id}")
        OUTPUT_DIR_GLOBAL = BASE_DIR / "aggregated_data"
        os.makedirs(OUTPUT_DIR_GLOBAL, exist_ok=True)
        
        global_single_dfs = []
        global_multi_dfs = []

        for day_code in sorted(ALL_DAYS):
            day_folder = BASE_DIR / day_code
            
            if not day_folder.exists():
                continue

            cprint(f"\n" + "-"*50, "cyan")
            cprint(f"📅 PROCESSING DAY: {day_code} in {dataset_id}", "cyan", attrs=["bold"])
            
            day_dfs = []

            # ---------------------------------------------------------------
            # A) PROCESS SINGLE FISH
            # ---------------------------------------------------------------
            dir_single = day_folder / "single_fish"
            
            if dir_single.exists() and day_code in single_fish_gt:
                cprint(f"\n🐟 Looking for SINGLE FISH...", "magenta")
                # Use the correct folder name for your case (assuming "results" for single fish too)
                df_single = rutils.aggregate_results_from_root_new(dir_single, day_code, output_csv_path=dir_single, results_foldername="results")
                
                if not df_single.empty:
                    medida = single_fish_gt[day_code]
                    especie = reverse_fish_measures.get(medida, "unknown_single_fish")
                    
                    df_single["gt"] = medida
                    df_single["especie_gt"] = especie
                    df_single["abs_not_a_fish_cm"] = abs((df_single["filteRed_tag_length"] * 100) - medida)
                    df_single["failure_reason"] = np.nan
                    df_single['unique_track'] = df_single['video_day'].astype(str) + "/" + df_single['video_name'].astype(str) + "/" + df_single['track_id'].astype(str)
                    df_single["scenario"] = "single_fish"
                    
                    df_single = rutils.assign_failure_reasons(df_single, aspect_ratio_thr=ASPECT_RATIO_THR, angle_thr=ANGLE_THR)
                    
                    out_single = dir_single / f"{day_code}_single_fish_agg.csv"
                    df_single.to_csv(out_single, index=False)
                    
                    global_single_dfs.append(df_single)
                    day_dfs.append(df_single) 
                    cprint(f"   ✅ Single Fish completed.", "green")
            else:
                if not dir_single.exists():
                    cprint(f"   ℹ️ No 'single_fish' folder for this day.", "dark_grey")
            
            # ---------------------------------------------------------------
            # B) PROCESS MULTIPLE FISH
            # ---------------------------------------------------------------
            dir_multi = day_folder / "multiple_fish"  
            
            if dir_multi.exists() and day_code in multiple_fish_gt:
                cprint(f"\n🐠🐠 Looking for MULTIPLE FISH...", "magenta")
                df_multi_raw = rutils.aggregate_results_from_root_new(dir_multi, day_code, output_csv_path=dir_multi, results_foldername="results")
                
                if not df_multi_raw.empty:
                    df_multi_raw['track_id'] = pd.to_numeric(df_multi_raw['track_id'], not_a_fishs='coerce').fillna(-1).astype(int)
                    
                    df_multi, huerfanos = rutils.inject_ground_truth(
                        df_multi_raw, 
                        {day_code: multiple_fish_gt[day_code]}, 
                        fish_measures, 
                        drop_unlabeled=False 
                    )
                    df_multi["scenario"] = "multiple_fish"
                    
                    df_multi = rutils.assign_failure_reasons(df_multi, aspect_ratio_thr=ASPECT_RATIO_THR, angle_thr=ANGLE_THR)
                    
                    out_multi = dir_multi / f"{day_code}_multiple_fish_agg.csv"
                    df_multi.to_csv(out_multi, index=False)
                    
                    global_multi_dfs.append(df_multi)
                    day_dfs.append(df_multi)
                    cprint(f"   ✅ Multiple Fish completed.", "green")
            else:
                if not dir_multi.exists():
                    cprint(f"   ℹ️ No 'multiple_fish' folder for this day.", "dark_grey")

            # ---------------------------------------------------------------
            #  C) SAVE DAILY AGGREGATE
            # ---------------------------------------------------------------
            if day_dfs:
                df_day_all = pd.concat(day_dfs, ignore_index=True)
                out_day = day_folder / f"{day_code}_ALL_raw_aggregated.csv"
                df_day_all.to_csv(out_day, index=False)
                cprint(f"   📁 Daily aggregate created: {out_day.name}", "cyan")
                
        # ===================================================================
        # 3. CREATE AGGREGATED DATASETS FOR THIS SPECIFIC DATASET FOLDER
        # ===================================================================
        cprint("\n" + "="*50, "blue")
        cprint(f"🌍 GENERATING GLOBALS IN: {OUTPUT_DIR_GLOBAL.name}", "white", "on_blue", attrs=["bold"])

        df_master_single = pd.DataFrame()
        df_master_multi = pd.DataFrame()

        if global_single_dfs:
            df_master_single = pd.concat(global_single_dfs, ignore_index=True)
            out = OUTPUT_DIR_GLOBAL / f"{dataset_id}_ALL_single_fish_all_days.csv"
            df_master_single.to_csv(out, index=False)
            cprint(f"📦 Saved: {out.name} ({len(df_master_single)} frames)", "cyan")

        if global_multi_dfs:
            df_master_multi = pd.concat(global_multi_dfs, ignore_index=True)
            out = OUTPUT_DIR_GLOBAL / f"{dataset_id}_ALL_multiple_fish_all_days.csv"
            df_master_multi.to_csv(out, index=False)
            cprint(f"📦 Saved: {out.name} ({len(df_master_multi)} frames)", "cyan")

        if not df_master_single.empty or not df_master_multi.empty:
            df_master_all = pd.concat([df_master_single, df_master_multi], ignore_index=True)
            
            # Vectorial Filters
            df_master_all = rutils.assign_failure_reasons(df_master_all, aspect_ratio_thr=ASPECT_RATIO_THR, angle_thr=ANGLE_THR)
            
            out = OUTPUT_DIR_GLOBAL / f"{dataset_id}_MASTER_all_fish_all_days.csv"
            df_master_all.to_csv(out, index=False)
            cprint(f"🏆 MASTER DATASET SAVED: {out.name} ({len(df_master_all)} frames)", "green", attrs=["bold"])

    cprint("\n🎉 ALL DATASETS COMPLETED SUCCESSFULLY!", "white", "on_magenta", attrs=["bold"])

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:

# ===================================================================
# 1. CONFIGURATION & PATHS
# ===================================================================
# Ensure these paths point to where the first script saved the MASTER files
DS1_MASTER_PATH = Path("/home/slimbook/fish_sizing/paper/results_csvs/DATASET_1/aggregated_data/DATASET_1_MASTER_all_fish_all_days.csv")
DS2_MASTER_PATH = Path("//home/slimbook/fish_sizing/paper/results_csvs/DATASET_2/aggregated_data/DATASET_2_MASTER_all_fish_all_days.csv")

# ===================================================================
# 2. DATA LOADING & TRACK AGGREGATION
# ===================================================================
df_ds1_final, df_ds1_raw= rutils.load_and_filter_dataset(DS1_MASTER_PATH, "Dataset_1",save=True)
df_ds2_final, df_ds2_raw = rutils.load_and_filter_dataset(DS2_MASTER_PATH, "Dataset_2",save=True)

df_all = pd.concat([df_ds1_final, df_ds2_final], ignore_index=True)
df_all_raw = pd.concat([df_ds1_raw, df_ds2_raw], ignore_index=True)

if df_all.empty:
    raise ValueError("No data was loaded. Check your paths!")

global_csv_path = "GLOBAL_FINAL_aggregated_metrics.csv"
df_all.to_csv(global_csv_path, index=False)
print(f"\n💾 Guardado CSV GLOBAL con todos los datos en: {global_csv_path}")

# ===================================================================
# 3. CREATING CATEGORIES FOR ANALYSIS & 4. METRICS TABLE
# ===================================================================
plot_data = []
summary_stats = []

def analyze_group(cat_name, dataset=None, scenario=None):
    # 1. Contar los tracks INICIALES desde el df crudo
    mask_raw = pd.Series(True, index=df_all_raw.index)
    if dataset: mask_raw &= (df_all_raw['dataset'] == dataset)
    if scenario: mask_raw &= (df_all_raw['scenario'] == scenario)
    
    sub_raw = df_all_raw[mask_raw]
    total_tracks = sub_raw['track_uid'].nunique() if not sub_raw.empty else 0
    
    # Calculate over only the tracks that are real fish ?
    # total_tracks = sub_raw[sub_raw['gt'] > 0]['track_uid'].nunique() if not sub_raw.empty else 0
    
    # 2. Contar los tracks FINALES (y métricas) desde el df procesado
    mask_final = pd.Series(True, index=df_all.index)
    if dataset: mask_final &= (df_all['dataset'] == dataset)
    if scenario: mask_final &= (df_all['scenario'] == scenario)
    
    sub_final = df_all[mask_final]
    measuRed_tag_tracks = len(sub_final)
    
    # 3. Calcular métricas si hay datos
    if measuRed_tag_tracks > 0:
        mae = sub_final['abs_error_cm'].mean()
        std = sub_final['abs_error_cm'].std()
        mape = sub_final['rel_error_perc'].mean()
    else:
        print("WEEEY")
        mae = None 
        mape = None
        
    # Guardar en la lista de resumen para imprimir la tabla luego
    summary_stats.append({
        'Category': cat_name,
        'Total Tracks': total_tracks,
        'MeasuRed_tag Tracks': measuRed_tag_tracks,
        'Success Rate (%)': (measuRed_tag_tracks/total_tracks*100) if total_tracks > 0 else 0,
        'MAE': mae,
        'STD': std,
        'MAPE': mape
    })
    
    # Guardar los datos del grupo para el boxplot
    if not sub_final.empty:
        temp = sub_final.copy()
        temp['Analysis_Group'] = cat_name
        plot_data.append(temp)

# Generamos las categorías dinámicamente
analyze_group("Dataset A\nSingle", dataset='Dataset_1', scenario='single_fish')
analyze_group("Dataset A\nMultiple", dataset='Dataset_1', scenario='multiple_fish')
analyze_group("Dataset A\nGlobal", dataset='Dataset_1')

analyze_group("Dataset B\nSingle", dataset='Dataset_2', scenario='single_fish')
analyze_group("Dataset B\nMultiple", dataset='Dataset_2', scenario='multiple_fish')
analyze_group("Dataset B\nGlobal", dataset='Dataset_2')

analyze_group("Global\nSingle", scenario='single_fish')
analyze_group("Global\nMultiple", scenario='multiple_fish')
analyze_group("GLOBAL\nTOTAL")

df_summary = pd.DataFrame(summary_stats)
df_plot = pd.concat(plot_data, ignore_index=True)

# IMPRIMIR LA TABLA FINAL
print("\n" + "="*104)
print(f"{'CATEGORY':<30} | {'Total Tracks':<12} | {'MeasuRed_tag':<10} | {'Success (%)':<12} | {'MAE (cm)':<10} | {'STD (cm)':<10} | {'MAPE (%)':<10}")
print("-" * 104)

for _, row in df_summary.iterrows():
    group_clean = row['Category'].replace("\n", " ")
    print(f"{group_clean:<30} | {row['Total Tracks']:<12} | {row['MeasuRed_tag Tracks']:<10} | {row['Success Rate (%)']:<12.1f} | {row['MAE']:<10.2f} | {row['STD']:<10.2f} | {row['MAPE']:<10.2f}")
print("="*104 + "\n")

# ===================================================================
# 5. BOXPLOT
# ===================================================================
plt.figure(figsize=(14, 7))
sns.set_theme(style="whitegrid", context="paper", font_scale=1.6)

palette = [
    "#9ECAE1", "#4292C6", "#08519C",  # DS1: Single, Multiple, Global
    "#FDD0A2", "#FD8D3C", "#D94801",  # DS2: Single, Multiple, Global
    "#BDBDBD", "#737373", "#252525"   # Globals: Single, Multiple, TOTAL
]

ax = sns.boxplot(
    data=df_plot, 
    x='Analysis_Group', 
    y='abs_error_cm', 
    palette=palette,
    showfliers=False,
    linewidth=1.5
)

plt.title("Absolute Sizing error Across Datasets and Scenarios", fontsize=16, fontweight='bold', pad=20)
plt.ylabel("Absolute error (cm)", fontsize=16, fontweight='bold')
plt.xlabel("", fontsize=16) 

plt.axhline(y=1.0, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Target error (1 cm)')
plt.legend()

plt.tight_layout()
plt.savefig("Final_Metrics_Boxplot.png", dpi=300)
plt.show()

# ===================================================================
# 6. METRICS & PLOT BY SPECIES (especie_gt) - UNIFICADO
# ===================================================================
if 'especie_gt' in df_all.columns:
    species_stats = []
    
    # 1. DICCIONARIO DE MAPEO (Traducción a nombres científicos/formales)
    species_mapping = {
        'Red_tag': 'D. labrax',
        'Black_tag': 'D. labrax',
        'unnmarked': 'D. labrax',
        'Fish_0': 'D. labrax',
        'Fish_3': 'D. labrax',
        
        'Fish_4': 'M. poutassou',
        'Fish_5': 'M. poutassou',
        'Fish_5': 'M. poutassou',
        
        'Fish_2': 'B. boops',
        'Fish_1': 'S. scombrus'
    }
    
    # 2. APLICAR EL MAPEO A AMBOS DATAFRAMES
    # Rellenamos NaNs primero, luego aplicamos el replace. 
    # Si alguna etiqueta no está en el diccionario, se quedará con su nombre original.
    df_all_raw['especie_gt_clean'] = df_all_raw['especie_gt'].fillna('Unknown').replace(species_mapping)
    df_all['especie_gt_clean'] = df_all['especie_gt'].fillna('Unknown').replace(species_mapping)
    
    # Sacamos las especies únicas ya unificadas
    unique_species = df_all_raw['especie_gt_clean'].unique()
    
    print("\n" + "="*105)
    print(" " * 35 + "METRICS BY SPECIES (UNIFIED)")
    print("="*105)
    print(f"{'SPECIES':<30} | {'Total Tracks':<12} | {'MeasuRed_tag':<10} | {'Success (%)':<12} | {'MAE (cm)':<10} | {'STD (cm)':<10} | {'MAPE (%)':<10}")
    print("-" * 105)
    
    for species in unique_species:
        # Total tracks crudos para esta especie unificada
        sub_raw = df_all_raw[df_all_raw['especie_gt_clean'] == species]
        total_s_tracks = sub_raw['track_uid'].nunique() if not sub_raw.empty else 0
        
        # Si quiero quitar los falsos positivos
        # total_s_tracks = sub_raw[sub_raw['gt'] > 0]['track_uid'].nunique() if not sub_raw.empty else 0
        
        # Tracks medidos y métricas para esta especie unificada
        sub_final = df_all[df_all['especie_gt_clean'] == species]
        measuRed_tag_s_tracks = len(sub_final)
        
        if measuRed_tag_s_tracks > 0:
            mae = sub_final['abs_error_cm'].mean()
            std = sub_final['abs_error_cm'].std()
            mape = sub_final['rel_error_perc'].mean()
        else:
            mae = std = mape = 0.0
            
        success_perc = (measuRed_tag_s_tracks/total_s_tracks*100) if total_s_tracks > 0 else 0
            
        print(f"{species:<30} | {total_s_tracks:<12} | {measuRed_tag_s_tracks:<10} | {success_perc:<12.1f} | {mae:<10.2f} | {std:<10.2f} | {mape:<10.2f}")
        
    print("="*105 + "\n")

    # ---- BOXPLOT POR ESPECIE UNIFICADA ----
    if len(df_all) > 0:
        plt.figure(figsize=(12, 6))
        sns.set_theme(style="whitegrid", context="paper", font_scale=1.6)
        
        # Ordenamos las especies por la mediana del error
        order = df_all.groupby('especie_gt_clean')['abs_error_cm'].median().sort_values().index
               
     
        ax2 = sns.boxplot(
            data=df_all, 
            x='especie_gt_clean', 
            y='abs_error_cm', 
            order=order,
            palette="ch:s=.25,rot=-.25", 
            showfliers=False,
            linewidth=1.5
        )
        
        plt.title("Absolute errorh Distribution by Species", fontsize=16, fontweight='bold', pad=20)
        plt.ylabel("Absolute error (cm)", fontsize=16, fontweight='bold')
        plt.xlabel("Species Ground Truth", fontsize=16, fontweight='bold')
        
        plt.axhline(y=1.0, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Target error (1 cm)')
        
        plt.xticks(rotation=30, ha='right') 
        
        
        plt.tight_layout()
        plt.savefig("Final_Metrics_Boxplot_Species.png", dpi=300)
        plt.show()
else:
    print("⚠️ No 'especie_gt' column found in the dataset. Skipping species analysis.")
    

In [ ]:
# ===================================================================
# 7. METRICS & PLOTS BY DATASET (Ablation, Geometry & Track Length)
# ===================================================================
datasets = df_all_raw['dataset'].unique()

for ds in datasets:
    cprint(f"\n\n{'='*70}", "white", "on_magenta", attrs=["bold"])
    cprint(f"📊 PLOTS FOR DATASET: {ds.upper()}", "white", "on_magenta", attrs=["bold"])
    cprint(f"{'='*70}", "white", "on_magenta", attrs=["bold"])
    
    
    df_ds_raw = df_all_raw[df_all_raw['dataset'] == ds].copy()
    
    
    df_ds_final = df_all[df_all['dataset'] == ds] if not df_all.empty else pd.DataFrame()
    
    context_label = f"GLOBAL - {ds.upper()}"
    

    # 7.2. Track Failure Distribution
    print(f"\n📈 Generating Track Failures Distribution Plot for {ds}...")
    rutils.plot_tracks_failure_distribution(
        df_ds_raw, 
        aspect_ratio_thr=ASPECT_RATIO_THR, 
        angle_thr=ANGLE_THR,
        show_global=True, 
        figsize_video=(12,6), 
        figsize_global=(5,5),
        context_label=context_label
    )


    


In [ ]:
def plot_global_failure_causes_stacked(df_all_raw, min_frames=5, save_path="Global_Failure_Causes_Stacked.png"):
    print("\n📊 Analyzing global track failure causes (Stacked Pipeline)...")


    failure_series = df_all_raw.groupby('track_uid').apply(
        lambda x: rutils.track_failure_from_frames(x, min_frames=min_frames)
    ).rename('Failure Reason')
    
    metadata = df_all_raw.groupby('track_uid').first()[['dataset', 'scenario']]
    
    track_df = pd.concat([failure_series, metadata], axis=1)
    
    reason_labels = {
        'measured': 'Successfully Measured',
        'short_track': f'Short Track (≤ {min_frames} frames)',
        'angle_fail': 'Elevation Angle above threshold',
        'aspect_ratio_fail': 'Aspect ratio below threshold',
        'overlap': 'Multiple Fish Overlap',
        'borders': 'Truncated by Image Border',
        'incomplete_3D': 'Incomplete 3D Pointcloud',
        'not_a_fish': 'False Positive / not_a_fish',
        'bad_pointcloud': 'Small or erroneous Pointcloud',
        'other': 'Other'
        
    
    }
    
    track_df['Failure Reason'] = track_df['Failure Reason'].map(reason_labels).fillna(track_df['Failure Reason'])
    
    track_df['Group'] = track_df['dataset'].astype(str).str.replace('_', ' ') + " (" + \
                        track_df['scenario'].astype(str).str.replace('_', ' ').str.title() + ")"
    

    pivot_df = track_df.groupby(['Failure Reason', 'Group']).size().unstack(fill_value=0)
    

    pivot_df['Total'] = pivot_df.sum(axis=1)
    pivot_df = pivot_df.sort_values(by='Total', ascending=True) 
    
    
    totals = pivot_df['Total'].copy()
    grand_total = totals.sum() 
    pivot_df = pivot_df.drop(columns='Total')
    
    # Colors
    color_map = {
        'Dataset 1 (Single Fish)': '#A9CCE3',   # Azul claro
        'Dataset 1 (Multiple Fish)': '#2471A3', # Azul oscuro
        'Dataset 2 (Single Fish)': '#F5CBA7',   # Naranja claro
        'Dataset 2 (Multiple Fish)': '#CA6F1E'  # Naranja oscuro
    }
    

    ordered_cols = [c for c in color_map.keys() if c in pivot_df.columns]
    pivot_df = pivot_df[ordered_cols]
    
    # ----------------- PLOT -----------------
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
    fig, ax = plt.subplots(figsize=(12, 7))
    

    pivot_df.plot(
        kind='barh', 
        stacked=True, 
        color=[color_map[c] for c in pivot_df.columns], 
        ax=ax, 
        edgecolor='black', 
        linewidth=0.5
    )
    
    # ----------------- LABELS -----------------
    plt.title("Primary Causes of Track Rejection by Dataset and Scenario", fontsize=15, fontweight='bold', pad=15)
    plt.xlabel("Total Number of Tracks", fontsize=12, fontweight='bold')
    plt.ylabel("")
    

    max_x = totals.max()
    for i, total in enumerate(totals):
        if total > 0:
            pct = (total / grand_total) * 100
            text_label = f"{int(total)} ({pct:.1f}%)"
            ax.text(total + (max_x * 0.015), i, text_label, va='center', fontweight='bold', fontsize=10, color='black')


    ax.set_xlim(0, max_x * 1.15)


    plt.legend(title="Data Source", bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)

    sns.despine(left=True, bottom=False)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    return pivot_df


pivot_df = plot_global_failure_causes_stacked(df_all_raw, min_frames=5)
print("\n=== RESUMEN TABULAR ===\n", pivot_df)